# Component 1 — Acoustic Robustness & Quality Gate

Component 1 is the **first stage** of the PD speech-screening pipeline. It is a
**quality gate, not a PD classifier**. For every recording it decides **PASS**
(forward the audio to the next stage) or **FAIL** (tell the user how to re-record).

`run_gate()` (`component1_robustness/pipeline.py`) runs, in order:

1. **Standardise** — any format → mono, at the recording's own sample rate
2. **Duration** — 1.5–5 s, otherwise `too_short` / `too_long`
3. **Quality measurements** — speech-band contrast, SNR/HNR, clipping, voice activity, MFCCs
4. **Enhancement** — 80 Hz high-pass (removes hum/rumble; chosen in section 9)
5. **Biomarker preservation** — did cleaning keep the receiving component's features? If not, the raw audio is used
6. **Gate model** — `comp1-B-svm` with a threshold tuned for ≤ 10 % good recordings rejected (section 8)
7. **Voice check** — a flat take (nobody speaking) → `no_voice` / `low_snr`
8. **Task check** — not one steady "aaaa" → `wrong_task`

This notebook only **calls** Component 1's code; it re-implements nothing, so what
you see is exactly what the CLI, the API (port 8001) and the app get.

| Section | What it shows |
|---|---|
| 1 | Setup and configuration |
| 2–4 | Vowel ("aaaa") walkthrough: input → one `run_gate()` call → inside the report |
| 5 | Vowel under different recording conditions (noise, silence, wrong task, …) |
| 6 | How the gate model and its threshold were chosen (benchmark) |
| 7 | How the enhancement method was chosen (vs the clean original) |
| 8 | DDK ("pa-ta-ka") through the gate |
| 9 | Personal recordings |

## 1. Setup and configuration

Run with the project's `.venv` as the kernel (needs `pip install notebook ipykernel`
besides `requirements.txt`). The first cell finds `parkinson-speech-ai/` wherever the
notebook is opened from.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

def find_project_root(start=None):
    """Walk up from the notebook folder until configs/component1.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "component1.yaml").is_file():
            return folder
    raise FileNotFoundError("Could not find parkinson-speech-ai/ (configs/component1.yaml). "
                            "Open this notebook from inside the project.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, Image, display

from component1_robustness.conditions import find_clips
from component1_robustness.config import load_config, resolve_path
from component1_robustness.degradation import add_white_noise, apply_gain
from component1_robustness.gate_models import load_gate_model
from component1_robustness.pipeline import TASKS, load_standardised, run_gate

print("Project root:", PROJECT_ROOT)

In [ ]:
CONFIG = load_config()   # configs/component1.yaml — fixed by the developer, never by the end user

settings = {
    "config_version": CONFIG["config_version"],
    "valid duration (s)": f'{CONFIG["min_duration_s"]} - {CONFIG["max_duration_s"]}',
    "max clipping (%)": CONFIG["max_clipping_pct"],
    "gate model": CONFIG["gate_model"],
    "gate pass probability (tuned)": CONFIG["gate_pass_probability"],
    "voice check": CONFIG.get("voice_check"),
    "task check": CONFIG.get("task_check"),
    "enhancement method": CONFIG["enhancement_method"],
    "enhancement policy": CONFIG.get("enhancement"),
    "feature tolerances (vowel)": CONFIG["feature_tolerances"],
    "feature tolerances (ddk)": CONFIG["ddk_feature_tolerances"],
    "training label rule": CONFIG.get("labels"),
}
with pd.option_context("display.max_colwidth", 200):
    display(pd.Series(settings, name="value").to_frame())

GATE = load_gate_model(CONFIG["gate_model"], CONFIG)   # loaded once, reused by every run_gate() below
print("Gate model loaded:", GATE.name)

Helpers used by both the vowel and the DDK walkthrough (written once here, not copied per section):

- `make_synthetic_pataka()` — a "pa-ta-ka"-like clip (5 short voiced bursts per second); the default DDK input, and the "wrong task" example for the vowel gate
- `show_report()` — the key fields of a `report.json`
- `preservation_table()` — each checked biomarker before / after cleaning vs its tolerance
- `run_conditions()` — damage a recording in controlled ways and run every version through the gate

In [ ]:
NB_DIR = PROJECT_ROOT / "outputs" / "component1_notebook"
NB_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_KEYS = ["clip_id", "task", "gate_model", "decision", "reason", "quality_score",
                "audio_choice", "next_component", "user_message", "config_version"]
QUALITY_KEYS = ["speech_band_contrast_db", "estimated_snr_db", "method", "clipping_pct",
                "voice_activity_pct", "level_dbfs", "duration_s"]


def make_synthetic_pataka(sr=16000, seconds=4.0, seed=CONFIG["seed"]):
    """150 Hz voiced bursts of 0.13 s, 5 per second, over a faint noise floor (SYNTHETIC)."""
    t = np.arange(int(seconds * sr)) / sr
    voiced = sum(np.sin(2 * np.pi * 150 * k * t) / k for k in range(1, 6))
    envelope = np.convolve(((t % 0.2) < 0.13).astype(float), np.ones(160) / 160, mode="same")
    y = 0.5 * voiced / np.max(np.abs(voiced)) * envelope
    y += np.random.default_rng(seed).standard_normal(len(y)) * np.sqrt(np.mean(y ** 2)) * 10 ** (-30 / 20)
    return 0.9 * y / np.max(np.abs(y)), sr


def show_report(report, title="report.json"):
    display(pd.Series({k: report[k] for k in SUMMARY_KEYS}, name=title).to_frame())


def preservation_table(report, tolerance_key):
    """Biomarkers before / after cleaning vs the allowed relative change."""
    pres = report["feature_preserved"]
    if pres is None:
        print("Rejected before the check:", report["reason"])
        return
    before, after = report["features_before"] or {}, report["features_after"] or {}
    rows = []
    for name, tol in CONFIG[tolerance_key].items():
        change = pres["per_feature"].get(name, float("nan"))
        rows.append({"feature": name, "before": before.get(name), "after": after.get(name),
                     "relative change": change, "allowed": tol,
                     "preserved": bool(change <= tol) if change == change else False})
    display(pd.DataFrame(rows).set_index("feature"))
    print("Preserved:", pres["preserved"], "  ->  audio forwarded:", report["audio_choice"])


def run_conditions(y, sr, task, extra=None, seed=CONFIG["seed"]):
    """The recording damaged in controlled ways, each version run through run_gate()."""
    rng = np.random.default_rng(seed)
    conditions = {
        "clean (as recorded)": y,
        "noise 20 dB": add_white_noise(y, 20, rng),
        "noise 10 dB": add_white_noise(y, 10, rng),
        "noise 5 dB": add_white_noise(y, 5, rng),
        "noise 0 dB": add_white_noise(y, 0, rng),
        "quiet (-12 dB gain)": apply_gain(y, -12),
        "clipped (too loud)": np.clip(y / (np.max(np.abs(y)) + 1e-12) * 4.0, -1.0, 1.0),
        "silence (nobody speaking)": rng.standard_normal(len(y)) * 10 ** (-50 / 20),
        "too short (1 s)": y[:int(1.0 * sr)],
        "too long": np.tile(y, int(np.ceil(CONFIG["max_duration_s"] * sr / len(y))) + 1),
        **(extra or {}),
    }
    out_dir = NB_DIR / f"conditions_{task}"
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for name, signal in conditions.items():
        peak = np.max(np.abs(signal))
        if not name.startswith("clipped") and peak > 0.99:   # keep non-clipping cases below full scale
            signal = 0.99 * signal / peak
        path = out_dir / (name.split(" (")[0].replace(" ", "_") + ".wav")
        sf.write(path, signal, sr)
        r = run_gate(path, task, CONFIG, gate=GATE)
        q = r["quality_before"] or {}
        rows.append({"condition": name, "contrast (dB)": q.get("speech_band_contrast_db"),
                     "decision": r["decision"], "reason": r["reason"],
                     "score": round(r["quality_score"], 1), "forwarded": r["audio_choice"],
                     "message to user": r["user_message"]})
    with pd.option_context("display.max_colwidth", 120, "display.float_format", "{:.1f}".format):
        display(pd.DataFrame(rows).set_index("condition"))

## 2. Input recording

One recording (`.wav`, `.m4a` or `.mp3`) and its **task**: `vowel` (a sustained "aaaa")
or `ddk` ("pa-ta-ka" repeated). The task decides which biomarkers the cleaning must preserve.

By default this uses the first **real healthy sustained vowel** from the AH dataset
(`data/vowel_task/raw/HC_AH`, see `data/vowel_task/README.md`) that fits the 1.5–5 s limit.
Set `INPUT_PATH` to use your own recording instead.

In [ ]:
INPUT_PATH = None   # e.g. PROJECT_ROOT / "data" / "personal_recordings" / "my_aaaa.wav"
TASK = "vowel"

if INPUT_PATH is None:
    healthy = [p for p, group in find_clips(resolve_path(CONFIG, "raw_data")) if group == "HC"]
    fits = [p for p in healthy if CONFIG["min_duration_s"] <= sf.info(p).duration <= CONFIG["max_duration_s"]]
    assert fits, "No AH dataset vowels found - extract them to data/vowel_task/raw (see data/vowel_task/README.md)"
    INPUT_PATH = fits[0]

INPUT_PATH = Path(INPUT_PATH)
assert INPUT_PATH.is_file(), f"Recording not found: {INPUT_PATH}"
assert TASK in TASKS, f"TASK must be one of {list(TASKS)}"

y_raw, sr = load_standardised(INPUT_PATH)   # the pipeline's own loader
print(f"File: {INPUT_PATH.name}   task: {TASK}   {sr} Hz   {len(y_raw) / sr:.2f} s")
display(Audio(y_raw, rate=sr))

## 3. The gate in one call

`run_gate()` runs every step listed at the top and writes
`outputs/component1/<clip_id>/` → `raw.wav`, `enhanced.wav`, `waveform.png`,
`spectrogram.png`, `report.json`. Running it twice on the same file and config gives a
byte-identical `report.json` (no timestamps).

In [ ]:
report = run_gate(INPUT_PATH, TASK, CONFIG, gate=GATE)
show_report(report)
OUT_DIR = resolve_path(CONFIG, "outputs") / report["clip_id"]
print("Output folder:", OUT_DIR, "->", sorted(f.name for f in OUT_DIR.iterdir()))

## 4. Inside the report

### 4a. Quality before / after cleaning, and what the enhancement did

`speech_band_contrast_db` (300–3400 Hz, loud vs quiet frames) is what `comp1-A` and the
voice check use; the `comp1-B` models also use the other measures and MFCCs.
Raw on top, cleaned below, same scales.

In [ ]:
if report["quality_before"] is None:
    print("Rejected by the duration rule before any measurement:", report["reason"])
else:
    display(pd.DataFrame({"raw": {k: report["quality_before"][k] for k in QUALITY_KEYS},
                          "cleaned": {k: report["quality_after"][k] for k in QUALITY_KEYS}}))
    if "enhancement" in report:
        display(pd.Series(report["enhancement"], name="enhancement").to_frame())
    display(Image(filename=str(OUT_DIR / "waveform.png")))
    display(Image(filename=str(OUT_DIR / "spectrogram.png")))

### 4b. Biomarker preservation (voice features)

Parkinson's shows up as small voice irregularities (jitter, shimmer, HNR). If cleaning
moved any checked feature beyond its tolerance, the **raw** audio is judged and forwarded,
so the gate never passes on audio whose biomarkers it changed.

In [ ]:
preservation_table(report, "feature_tolerances")

### 4c. Voice check and task check

- **Voice check** (`voice_check.py`): a take flat in the speech band means nobody started
  or stopped speaking → `no_voice` (quiet room) or `low_snr` (loud steady noise).
- **Task check** (`task_check.py`, vowel only): one unbroken voiced stretch ≥ 1.0 s that is
  ≥ 50 % of all voicing. Calibrated on the 81 AH vowels (none rejected); speech or
  gibberish → `wrong_task`.

In [ ]:
checks = {"voice_check": report.get("voice_check"), "task_check": report.get("task_check")}
for name, values in checks.items():
    if values is None:
        print(f"{name}: not run (rejected earlier, or disabled in the config)")
    else:
        display(pd.Series(values, name=name).to_frame())

## 5. Vowel under different recording conditions

Expected: clean and mild noise pass; heavy noise, clipping, silence, a too-short/too-long
take and a **wrong task** (a "pa-ta-ka"-like clip given as "aaaa") are rejected with a
clear message.

In [ ]:
pataka, pataka_sr = make_synthetic_pataka(sr)
run_conditions(y_raw, sr, TASK, extra={"wrong task (pa-ta-ka-like)": pataka[:len(y_raw)]})

## 6. How the gate model and its threshold were chosen

`python -m component1_robustness.labels` then `python -m component1_robustness.benchmark`:

- **Labels** (`labels.py`, config `labels`): each of 80 participants × 12 degraded versions
  is labelled *reliable* or *unreliable* (rule and threshold in config `labels`).
- **Benchmark** (`benchmark.py`): participant-grouped 5-fold CV + a held-out test set.
  Priority: **good recordings must pass** — each model's threshold is tuned to reject
  ≤ 10 % of good clips, then the model letting the fewest bad clips through wins.

In [ ]:
results_dir = resolve_path(CONFIG, "results")
bench_csv, selected_json = results_dir / "gate_benchmark.csv", results_dir / "selected_gate_model.json"
if bench_csv.is_file() and selected_json.is_file():
    bench = pd.read_csv(bench_csv)
    cols = ["model", "label_source", "threshold", "accuracy_mean", "false_reject_rate_mean",
            "false_accept_rate_mean", "test_false_reject_rate", "test_false_accept_rate"]
    display(bench[[c for c in cols if c in bench.columns]].round(3))
    selected = json.loads(selected_json.read_text(encoding="utf-8"))
    print("Selected:", selected["selected_gate_model"])
    print("Why     :", selected["reason"])
    for where, t in (selected.get("targets") or {}).items():
        display(pd.Series(t, name=f"targets ({where})").to_frame())
else:
    print("Not found - run labels.py, then benchmark.py")

## 7. How the enhancement method was chosen

`python -m component1_robustness.compare_enhancement`: the 81 real AH
vowels get a 0.6 s lead-in, are degraded (clean, white noise 10/5/0 dB, 50 Hz hum +
rumble), cleaned by every method, and compared with the **clean original**.
A good method brings the numbers back towards clean and never harms a clean take.
The old `noisereduce` non-stationary default cut the voice level by ~10–12 dB on every
take; the 80 Hz high-pass leaves clean takes untouched and removes hum/rumble completely.

In [ ]:
cmp_csv = resolve_path(CONFIG, "results") / "enhancement_comparison_summary.csv"
if cmp_csv.is_file():
    cmp_df = pd.read_csv(cmp_csv)
    cols = ["condition", "method", "si_sdr_db", "level_db_error", "jitter_pct_error",
            "shimmer_pct_error", "hnr_db_error"]
    display(cmp_df[cols].round(2).set_index(["condition", "method"]))
    cmp_df.pivot(index="method", columns="condition", values="si_sdr_db").plot.barh(figsize=(9, 5))
    plt.xlabel("SI-SDR vs clean original (dB, higher = closer to clean)")
    plt.title("Enhancement methods vs the clean recording")
    plt.tight_layout(); plt.show()
else:
    print("Not found - run: python -m component1_robustness.compare_enhancement")

## 8. DDK ("pa-ta-ka") through the gate

For a DDK take the biomarkers are **rhythm** (syllable rate, gaps between syllables,
regularity, pauses), so the preservation check uses rhythm features and their own
tolerances (`ddk_feature_tolerances`). There is no task check for DDK yet. By default a
**synthetic** pa-ta-ka-like clip is used; set `DDK_PATH` to a real recording.

In [ ]:
DDK_PATH = None   # e.g. PROJECT_ROOT / "data" / "personal_recordings" / "my_pataka.wav"

if DDK_PATH is None:
    DDK_PATH = NB_DIR / "synthetic_pataka.wav"
    sf.write(DDK_PATH, *make_synthetic_pataka())
    print("Using a SYNTHETIC pa-ta-ka-like clip.")

y_ddk, sr_ddk = load_standardised(Path(DDK_PATH))
report_ddk = run_gate(DDK_PATH, "ddk", CONFIG, gate=GATE)
display(Audio(y_ddk, rate=sr_ddk))
show_report(report_ddk, "report.json (ddk)")
preservation_table(report_ddk, "ddk_feature_tolerances")

In [ ]:
run_conditions(y_ddk, sr_ddk, "ddk", seed=CONFIG["seed"] + 1)

## 9. Personal recordings

Put your own takes in `data/personal_recordings/` (`.wav`, `.m4a`, `.mp3`) and set the task.
Each runs through the gate exactly like an app upload. They are **never** used for
training, thresholds or evaluation.

In [ ]:
PERSONAL_DIR = PROJECT_ROOT / "data" / "personal_recordings"
PERSONAL_TASK = "vowel"   # or "ddk"
PERSONAL_DIR.mkdir(parents=True, exist_ok=True)

personal = sorted(p for p in PERSONAL_DIR.iterdir() if p.suffix.lower() in (".wav", ".m4a", ".mp3"))
rows = []
for path in personal:
    r = run_gate(path, PERSONAL_TASK, CONFIG, gate=GATE)
    rows.append({"file": path.name, "decision": r["decision"], "reason": r["reason"],
                 "score": round(r["quality_score"], 1), "forwarded": r["audio_choice"],
                 "message to user": r["user_message"]})
if rows:
    display(pd.DataFrame(rows).set_index("file"))
else:
    print(f"No recordings in {PERSONAL_DIR} - add some and re-run this cell.")

## Summary

- **Input:** one recording + task (`vowel` / `ddk`).
- **Process:** standardise → duration → quality measurements → 80 Hz high-pass →
  biomarker preservation → gate model (`comp1-B-svm`, threshold tuned for ≤ 10 % good
  recordings rejected) → voice check → task check.
- **Output:** PASS/FAIL, quality score, a message for the user, `report.json`, and the
  audio forwarded to the next stage (cleaned only if the biomarkers survived).
- **Evidence:** gate benchmark (section 6), enhancement comparison (section 7).
- **Verification:** `python -m pytest component1_robustness/tests -q`.

Known limitations are listed in `component1_robustness/README.md`.